# 18 - Dense Segmentation Pseudo-Labels for Temporal XAI

This notebook generates compact LV segmentation pseudo-labels for EchoNet-Dynamic frames needed by temporal Grad-CAM evaluation, anatomical overlap metrics, and optical-flow analysis.

The trained bidirectional ConvLSTM U-Net predicts only the center frame of a temporal sequence. For each frame that needs a mask, this notebook builds a shifted temporal window so that frame is at the model center, runs segmentation inference, and saves the resulting binary mask. If a ground-truth ED/ES mask exists for that frame, the ground-truth mask replaces the pseudo-label.

No Grad-CAMs, overlays, PNGs, or duplicate arrays are saved.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import os
import sys
from typing import Iterable

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Kaggle-compatible project discovery. PROJECT_ROOT should be the dataset root
# that contains the src/ package. If your Kaggle code dataset points directly at
# a folder named src, set PROJECT_ROOT to that folder's parent.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/kaggle/input/datasets/sooahnoh/echonet-src-code-3"))
if not (PROJECT_ROOT / "src").exists():
    fallback = Path.cwd()
    if (fallback / "src").exists():
        PROJECT_ROOT = fallback
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import BidirectionalConvLSTMUNet, build_bidirectional_convlstm_unet
from src.dataset import load_temporal_metadata, split_by_echonet_filelist
from src.utils import load_echonet_tables, normalize_video_name, set_seed


In [ ]:
RUN_MODE = "smoke"  # change to "full" after smoke mode completes

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", "/kaggle/input/datasets/jiyoonoh24/echonet-dynamic/EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", "/kaggle/input/datasets/jiyoonoh24/echonet-dynamic-processed-masks"))
VIDEOS_DIR = RAW_DIR / "Videos"

SEGMENTATION_MODEL_RUN_DIR = Path(
    os.environ.get(
        "SEGMENTATION_MODEL_RUN_DIR",
        "/kaggle/input/datasets/sooahnoh/multitask-segmentation-ef" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "multitask_segmentation_ef_07_20",
    )
)
OUTPUT_DIR = Path("/kaggle/working/outputs/runs/segmentation_pseudolabels") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "segmentation_pseudolabels"
MASK_DIR = OUTPUT_DIR / "masks_by_video"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
for directory in [OUTPUT_DIR, MASK_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": 42,
    "label_scope": "required_by_notebook05",  # "required_by_notebook05" or "full_test_videos"
    "notebook05_sequence_length": 5,
    "notebook05_temporal_strides": [1, 4, 6, 8, 10],
    "num_frames_before": 11,
    "num_frames_after": 11,
    "temporal_stride": 2,
    "image_size": [112, 112],
    "channels": [16, 32, 64, 128],
    "model_family": "multitask_segmentation_ef",  # "multitask_segmentation_ef" or "segmentation_only"
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "segmentation_threshold": 0.5,
    "batch_size": 8,
    "num_workers": 0,
    "smoke_video_count": 2,
    "save_probabilities": False,  # keep False for compact outputs; masks are enough for later ROI metrics
}

set_seed(int(CONFIG["seed"]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Segmentation model run directory: {SEGMENTATION_MODEL_RUN_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## Load EchoNet Tables and Ground-Truth Mask Lookup

The processed metadata contains one row per labeled ED/ES frame. Those masks are used as ground truth whenever the requested pseudo-label frame matches a labeled frame.


In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert VIDEOS_DIR.exists(), f"Missing EchoNet Videos directory: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, volume_tracings = load_echonet_tables(RAW_DIR)
train_samples, validation_samples, test_samples = split_by_echonet_filelist(samples, file_list)
assert test_samples, "No processed ED/ES samples matched the official EchoNet test split."

# Ground-truth masks are keyed by exact video/frame.
gt_mask_lookup: dict[tuple[str, int], str] = {}
for sample in samples:
    gt_mask_lookup[(str(sample["video_id"]), int(sample["frame_idx"]))] = str(sample["mask"])

test_video_ids = sorted({str(sample["video_id"]) for sample in test_samples})
print({
    "processed_samples": len(samples),
    "test_ed_es_samples": len(test_samples),
    "test_videos_with_processed_masks": len(test_video_ids),
    "ground_truth_mask_entries": len(gt_mask_lookup),
})


## Discover and Load Segmentation Model Checkpoint

By default this notebook loads the segmentation-primary multitask checkpoint from notebook 11, selected by validation Dice (`best_model.pt` / `best_val_dice_model.pt`). The EF output is ignored; only segmentation logits are used for pseudo-labels.


In [ ]:
class MultiTaskBidirectionalConvLSTMUNet(BidirectionalConvLSTMUNet):
    """Notebook-11 segmentation-primary multitask model.

    The checkpoint contains both the segmentation path and EF head. Pseudo-label
    generation uses only the first output, ``seg_logits``.
    """

    def __init__(self, *args, ef_hidden_dim: int = 128, dropout: float = 0.1, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        bottleneck_channels = self.bidirectional_fusion[0].out_channels
        self.ef_pool = nn.AdaptiveAvgPool2d(1)
        self.ef_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(bottleneck_channels, ef_hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ef_hidden_dim, 1),
        )

    def forward(self, sequence: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        self._validate_sequence(sequence)
        bottlenecks, target_skips = self._encode_sequence(sequence)
        forward_hidden = self._run_temporal_branch(
            self.forward_temporal_bottleneck,
            bottlenecks,
            range(0, self.target_idx + 1),
        )
        backward_hidden = self._run_temporal_branch(
            self.backward_temporal_bottleneck,
            bottlenecks,
            range(self.expected_sequence_length - 1, self.target_idx - 1, -1),
        )
        fused_temporal = self.bidirectional_fusion(torch.cat([forward_hidden, backward_hidden], dim=1))
        ef_normalized = self.ef_head(self.ef_pool(fused_temporal)).squeeze(1)
        skip1, skip2, skip3 = target_skips
        x = self.decoder3(fused_temporal, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)
        seg_logits = self.output(x)
        return seg_logits, ef_normalized


def first_existing(paths: Iterable[Path | str | None]) -> Path | None:
    for candidate in paths:
        if candidate is None or str(candidate).strip() == "":
            continue
        path = Path(candidate)
        if path.exists():
            return path
    return None


def state_dict_from_checkpoint(checkpoint):
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint)) if isinstance(checkpoint, dict) else checkpoint
    if any(key.startswith("module.") for key in state_dict):
        state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
    return state_dict


def segmentation_logits_from_output(output):
    if isinstance(output, tuple):
        return output[0]
    if isinstance(output, dict):
        for key in ["seg_logits", "segmentation_logits", "segmentation", "mask_logits"]:
            if key in output:
                return output[key]
        raise KeyError(f"Could not find segmentation logits in output keys: {sorted(output)}")
    return output

config_path = first_existing([
    SEGMENTATION_MODEL_RUN_DIR / "config.json",
    SEGMENTATION_MODEL_RUN_DIR / "checkpoints" / "config.json",
])
if config_path is not None:
    trained_config = json.loads(config_path.read_text())
    for key in [
        "num_frames_before", "num_frames_after", "temporal_stride", "image_size", "channels",
        "segmentation_threshold", "threshold", "ef_hidden_dim", "dropout",
    ]:
        if key in trained_config:
            target_key = "segmentation_threshold" if key == "threshold" else key
            CONFIG[target_key] = trained_config[key]

checkpoint_path = first_existing([
    os.environ.get("SEGMENTATION_CHECKPOINT_PATH"),
    SEGMENTATION_MODEL_RUN_DIR / "checkpoints" / "best_model.pt",
    SEGMENTATION_MODEL_RUN_DIR / "checkpoints" / "best_val_dice_model.pt",
    SEGMENTATION_MODEL_RUN_DIR / "best_model.pt",
    SEGMENTATION_MODEL_RUN_DIR / "best_val_dice_model.pt",
    SEGMENTATION_MODEL_RUN_DIR / "checkpoints" / "final_model.pt",
])
assert checkpoint_path is not None and checkpoint_path.exists(), f"Could not find segmentation/multitask checkpoint under {SEGMENTATION_MODEL_RUN_DIR}"

if CONFIG.get("model_family", "multitask_segmentation_ef") == "multitask_segmentation_ef":
    model = MultiTaskBidirectionalConvLSTMUNet(
        in_channels=1,
        out_channels=1,
        channels=tuple(CONFIG["channels"]),
        num_frames_before=int(CONFIG["num_frames_before"]),
        num_frames_after=int(CONFIG["num_frames_after"]),
        ef_hidden_dim=int(CONFIG.get("ef_hidden_dim", 128)),
        dropout=float(CONFIG.get("dropout", 0.1)),
    ).to(device)
else:
    model = build_bidirectional_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=tuple(CONFIG["channels"]),
        num_frames_before=int(CONFIG["num_frames_before"]),
        num_frames_after=int(CONFIG["num_frames_after"]),
    ).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
state_dict = state_dict_from_checkpoint(checkpoint)
load_result = model.load_state_dict(state_dict, strict=True)
model.eval()

CONFIG["sequence_length"] = int(CONFIG["num_frames_before"]) + 1 + int(CONFIG["num_frames_after"])
CONFIG["target_idx"] = int(CONFIG["num_frames_before"])
CONFIG["checkpoint_path"] = str(checkpoint_path)
CONFIG["segmentation_model_run_dir"] = str(SEGMENTATION_MODEL_RUN_DIR)
CONFIG["run_mode"] = RUN_MODE
with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(CONFIG, file, indent=2)

print(json.dumps({
    "model_family": CONFIG.get("model_family"),
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_epoch": checkpoint.get("epoch") if isinstance(checkpoint, dict) else None,
    "checkpoint_metrics": checkpoint.get("metrics") if isinstance(checkpoint, dict) else None,
    "config_path": str(config_path) if config_path else None,
    "sequence_length": CONFIG["sequence_length"],
    "target_idx": CONFIG["target_idx"],
    "image_size": CONFIG["image_size"],
    "temporal_stride": CONFIG["temporal_stride"],
}, indent=2))


## Determine Which Frames Need Masks

`required_by_notebook05` reproduces the frame requirements from `notebooks/05_gradcam_temporal_evaluation.ipynb`: official test ED/ES samples, sequence length 5, and strides `[1, 4, 6, 8, 10]`. This is much smaller than segmenting every frame in every test video and is the recommended Kaggle setting.

`full_test_videos` is available if you later want every frame from every official test video, but it is much slower.


In [ ]:
def video_path(video_id: str) -> Path:
    filename = video_id if video_id.lower().endswith(".avi") else f"{video_id}.avi"
    return VIDEOS_DIR / filename


def video_frame_count(video_id: str) -> int:
    cap = cv2.VideoCapture(str(video_path(video_id)))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path(video_id)}")
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if count <= 0:
        raise ValueError(f"Video reports no frames: {video_path(video_id)}")
    return count


def official_test_video_ids_from_filelist(file_list: pd.DataFrame) -> list[str]:
    rows = file_list[file_list["Split"].astype(str).str.upper() == "TEST"]
    return sorted(Path(normalize_video_name(name)).stem for name in rows["FileName"].astype(str))


def required_frames_from_notebook05(samples_for_eval: list[dict[str, str | int]]) -> dict[str, set[int]]:
    required: dict[str, set[int]] = defaultdict(set)
    radius = int(CONFIG["notebook05_sequence_length"]) // 2
    frame_count_cache: dict[str, int] = {}
    for sample in samples_for_eval:
        video_id = str(sample["video_id"])
        center = int(sample["frame_idx"])
        if video_id not in frame_count_cache:
            frame_count_cache[video_id] = video_frame_count(video_id)
        frame_count = frame_count_cache[video_id]
        for stride in CONFIG["notebook05_temporal_strides"]:
            for rel in range(-radius, radius + 1):
                idx = min(max(center + rel * int(stride), 0), frame_count - 1)
                required[video_id].add(int(idx))
    return required


def full_frames_for_test_videos(video_ids: list[str]) -> dict[str, set[int]]:
    required: dict[str, set[int]] = {}
    for video_id in tqdm(video_ids, desc="count test video frames"):
        frame_count = video_frame_count(video_id)
        required[video_id] = set(range(frame_count))
    return required

if CONFIG["label_scope"] == "full_test_videos":
    target_video_ids = official_test_video_ids_from_filelist(file_list)
    required_frames_by_video = full_frames_for_test_videos(target_video_ids)
elif CONFIG["label_scope"] == "required_by_notebook05":
    required_frames_by_video = required_frames_from_notebook05(test_samples)
else:
    raise ValueError(f"Unknown label_scope: {CONFIG['label_scope']}")

if RUN_MODE == "smoke":
    keep_videos = sorted(required_frames_by_video)[: int(CONFIG["smoke_video_count"])]
    required_frames_by_video = {video_id: required_frames_by_video[video_id] for video_id in keep_videos}

planned_rows = []
for video_id, frame_set in required_frames_by_video.items():
    planned_rows.append({"video_id": video_id, "required_frame_count": len(frame_set), "min_frame_idx": min(frame_set), "max_frame_idx": max(frame_set)})
planned_df = pd.DataFrame(planned_rows).sort_values("video_id").reset_index(drop=True)
planned_df.to_csv(MANIFEST_DIR / "planned_frames_by_video.csv", index=False)
print({
    "run_mode": RUN_MODE,
    "label_scope": CONFIG["label_scope"],
    "videos_to_process": len(required_frames_by_video),
    "target_masks_to_save": int(sum(len(v) for v in required_frames_by_video.values())),
})
display(planned_df.head())


## Pseudo-Label Dataset and Compact Mask Saving

Masks are packed with `np.packbits`, so each binary `112 x 112` mask takes about 1.6 KB before zip compression instead of 12.5 KB. Later notebooks can recover masks with `np.unpackbits(..., axis=-1)[:, :, :W]`.


In [ ]:
class ShiftedCenterFrameDataset(Dataset):
    def __init__(self, video_id: str, frame_indices: list[int], image_size: tuple[int, int], offsets: list[int]):
        self.video_id = video_id
        self.frame_indices = list(map(int, frame_indices))
        self.image_size = image_size
        self.offsets = list(map(int, offsets))
        self.path = video_path(video_id)
        self.frame_count = video_frame_count(video_id)

    def __len__(self) -> int:
        return len(self.frame_indices)

    def _read_sequence(self, target_frame_idx: int) -> tuple[np.ndarray, np.ndarray]:
        cap = cv2.VideoCapture(str(self.path))
        if not cap.isOpened():
            raise FileNotFoundError(f"Could not open video: {self.path}")
        frames = []
        sampled_indices = []
        for offset in self.offsets:
            frame_idx = min(max(int(target_frame_idx) + offset, 0), self.frame_count - 1)
            sampled_indices.append(frame_idx)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame_bgr = cap.read()
            if not ok or frame_bgr is None:
                cap.release()
                raise ValueError(f"Could not read frame {frame_idx} from {self.path}")
            gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, (self.image_size[1], self.image_size[0]), interpolation=cv2.INTER_AREA)
            frames.append(gray.astype(np.float32) / 255.0)
        cap.release()
        return np.stack(frames, axis=0), np.asarray(sampled_indices, dtype=np.int32)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | int | str]:
        target_frame_idx = self.frame_indices[idx]
        sequence, sampled_indices = self._read_sequence(target_frame_idx)
        return {
            "sequence": torch.from_numpy(sequence).unsqueeze(1).contiguous(),
            "target_frame_idx": int(target_frame_idx),
            "sampled_frame_indices": torch.from_numpy(sampled_indices),
            "video_id": self.video_id,
            "frame_count": int(self.frame_count),
        }


def load_ground_truth_mask(video_id: str, frame_idx: int, image_size: tuple[int, int]) -> np.ndarray | None:
    mask_path = gt_mask_lookup.get((video_id, int(frame_idx)))
    if mask_path is None:
        return None
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not read ground-truth mask: {mask_path}")
    mask = cv2.resize(mask, (image_size[1], image_size[0]), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.uint8)


def pack_masks(mask_stack: np.ndarray) -> np.ndarray:
    return np.packbits(mask_stack.astype(np.uint8), axis=-1)


def mask_file_size_bytes() -> int:
    return sum(path.stat().st_size for path in MASK_DIR.glob("*.npz"))


## Generate Pseudo-Labels

This cell is the main work. It saves one `.npz` per video and two manifests:

- `pseudolabel_frame_manifest.csv`: one row per saved frame mask
- `pseudolabel_video_manifest.csv`: one row per video file


In [ ]:
offsets = [
    rel * int(CONFIG["temporal_stride"])
    for rel in range(-int(CONFIG["num_frames_before"]), int(CONFIG["num_frames_after"]) + 1)
]
image_size = tuple(map(int, CONFIG["image_size"]))
threshold = float(CONFIG["segmentation_threshold"])

frame_manifest_rows = []
video_manifest_rows = []
processed_video_count = 0
generated_pseudolabel_count = 0
ground_truth_substitution_count = 0

for video_id in tqdm(sorted(required_frames_by_video), desc="videos"):
    frame_indices = sorted(map(int, required_frames_by_video[video_id]))
    dataset = ShiftedCenterFrameDataset(video_id, frame_indices, image_size=image_size, offsets=offsets)
    loader = DataLoader(dataset, batch_size=int(CONFIG["batch_size"]), shuffle=False, num_workers=int(CONFIG["num_workers"]))

    saved_masks = []
    saved_probs = []
    saved_frame_indices = []
    saved_is_ground_truth = []
    saved_sequence_indices = []
    saved_source_mask_paths = []

    with torch.inference_mode():
        for batch in tqdm(loader, desc=f"segment {video_id}", leave=False):
            sequence = batch["sequence"].to(device)
            output = model(sequence)
            logits = segmentation_logits_from_output(output)
            probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
            target_frame_indices = batch["target_frame_idx"].detach().cpu().numpy().astype(np.int32)
            sequence_indices = batch["sampled_frame_indices"].detach().cpu().numpy().astype(np.int32)

            for item_idx, frame_idx in enumerate(target_frame_indices):
                gt_mask = load_ground_truth_mask(video_id, int(frame_idx), image_size=image_size)
                if gt_mask is not None:
                    binary_mask = gt_mask
                    is_gt = True
                    source_mask_path = gt_mask_lookup[(video_id, int(frame_idx))]
                    ground_truth_substitution_count += 1
                else:
                    binary_mask = (probs[item_idx] >= threshold).astype(np.uint8)
                    is_gt = False
                    source_mask_path = ""
                    generated_pseudolabel_count += 1

                saved_masks.append(binary_mask)
                if bool(CONFIG["save_probabilities"]):
                    saved_probs.append(probs[item_idx].astype(np.float16))
                saved_frame_indices.append(int(frame_idx))
                saved_is_ground_truth.append(bool(is_gt))
                saved_sequence_indices.append(sequence_indices[item_idx])
                saved_source_mask_paths.append(source_mask_path)

    if not saved_masks:
        continue

    mask_stack = np.stack(saved_masks, axis=0).astype(np.uint8)
    sequence_index_stack = np.stack(saved_sequence_indices, axis=0).astype(np.int32)
    out_path = MASK_DIR / f"{video_id}_segmentation_masks.npz"
    payload = {
        "video_id": np.asarray(video_id),
        "frame_indices": np.asarray(saved_frame_indices, dtype=np.int32),
        "masks_packed": pack_masks(mask_stack),
        "mask_shape": np.asarray(mask_stack.shape, dtype=np.int32),
        "is_ground_truth": np.asarray(saved_is_ground_truth, dtype=bool),
        "sequence_frame_indices": sequence_index_stack,
        "source_mask_paths": np.asarray(saved_source_mask_paths, dtype=str),
        "frame_count": np.asarray(dataset.frame_count, dtype=np.int32),
        "image_size": np.asarray(image_size, dtype=np.int32),
        "offsets": np.asarray(offsets, dtype=np.int32),
        "target_idx": np.asarray(CONFIG["target_idx"], dtype=np.int32),
        "temporal_stride": np.asarray(CONFIG["temporal_stride"], dtype=np.int32),
        "segmentation_threshold": np.asarray(threshold, dtype=np.float32),
    }
    if bool(CONFIG["save_probabilities"]):
        payload["probabilities_float16"] = np.stack(saved_probs, axis=0).astype(np.float16)
    np.savez_compressed(out_path, **payload)

    for row_idx, frame_idx in enumerate(saved_frame_indices):
        frame_manifest_rows.append({
            "video_id": video_id,
            "frame_idx": int(frame_idx),
            "mask_npz_path": str(out_path),
            "mask_row_index": int(row_idx),
            "is_ground_truth": bool(saved_is_ground_truth[row_idx]),
            "source_mask_path": saved_source_mask_paths[row_idx],
            "frame_count": int(dataset.frame_count),
            "sequence_frame_indices": json.dumps(sequence_index_stack[row_idx].tolist()),
            "label_scope": CONFIG["label_scope"],
        })

    video_manifest_rows.append({
        "video_id": video_id,
        "mask_npz_path": str(out_path),
        "saved_mask_count": int(len(saved_frame_indices)),
        "ground_truth_mask_count": int(np.asarray(saved_is_ground_truth, dtype=bool).sum()),
        "pseudo_label_count": int(len(saved_frame_indices) - np.asarray(saved_is_ground_truth, dtype=bool).sum()),
        "frame_count": int(dataset.frame_count),
        "storage_bytes": int(out_path.stat().st_size),
    })
    processed_video_count += 1

frame_manifest_df = pd.DataFrame(frame_manifest_rows)
video_manifest_df = pd.DataFrame(video_manifest_rows)
frame_manifest_df.to_csv(MANIFEST_DIR / "pseudolabel_frame_manifest.csv", index=False)
video_manifest_df.to_csv(MANIFEST_DIR / "pseudolabel_video_manifest.csv", index=False)

summary = {
    "run_mode": RUN_MODE,
    "label_scope": CONFIG["label_scope"],
    "processed_videos": int(processed_video_count),
    "saved_masks": int(len(frame_manifest_df)),
    "generated_pseudolabels": int(generated_pseudolabel_count),
    "ground_truth_masks_substituted": int(ground_truth_substitution_count),
    "output_storage_bytes": int(mask_file_size_bytes()),
    "output_storage_gb": float(mask_file_size_bytes() / (1024 ** 3)),
    "mask_storage_format": "np.savez_compressed with np.packbits binary masks",
    "checkpoint_path": str(checkpoint_path),
}
with (OUTPUT_DIR / "pseudolabel_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
display(video_manifest_df.head())
display(frame_manifest_df.head())


## Loading Example for Later Notebooks

This small example shows how to recover masks from a per-video NPZ file. It is not required for generation, but documents the storage format for future analysis notebooks.


In [ ]:
def load_video_masks_from_npz(path: str | Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    with np.load(path, allow_pickle=False) as data:
        shape = tuple(data["mask_shape"].astype(int).tolist())
        packed = data["masks_packed"]
        masks = np.unpackbits(packed, axis=-1)[..., : shape[-1]].reshape(shape).astype(bool)
        frame_indices = data["frame_indices"].astype(np.int32)
        is_ground_truth = data["is_ground_truth"].astype(bool)
    return masks, frame_indices, is_ground_truth

if not video_manifest_df.empty:
    example_path = video_manifest_df.iloc[0]["mask_npz_path"]
    masks, frame_indices, is_gt = load_video_masks_from_npz(example_path)
    print({
        "example_path": example_path,
        "masks_shape": masks.shape,
        "frame_indices_shape": frame_indices.shape,
        "ground_truth_count": int(is_gt.sum()),
    })
